# Functions


In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

# DATA LOADING
current_dir = Path.cwd()
project_root = current_dir.parents[2]
path_data = project_root / 'DATA_PPMI/Results/MODEL_DATA/DATA_test2.csv'
data = pd.read_csv(path_data,index_col=0)

# FUNCIONES PARA CALCULAR DELTA Y CLASIFICACIÓN HY3

def compute_delta(df_prev, df_next,sufix):

    # Alinear por index (inner join automático)
    df_prev, df_next = df_prev.align(df_next, join="inner", axis=0)
    
    # Seleccionar solo columnas numéricas
    numeric_cols = df_prev.select_dtypes(include='number').columns
    
    # Calcular delta
    delta = df_next[numeric_cols] - df_prev[numeric_cols]
    
    # Renombrar columnas
    delta.columns = [f"{col}_delta_{sufix}" for col in numeric_cols]
    
    return delta

def HY3_classification(nhy):
    if nhy == 0:
        return 0  # Healthy
    elif nhy  in [1,2]:
        return 1  # Early Stage
    else:
        return 2  # Advanced Stage
    
def HY4_classification(nhy):
    if nhy == 0:
        return 0  # Healthy
    elif nhy  == 1:
        return 1  # Early Stage
    elif nhy == 2:
        return 2  # Mid Stage
    else:
        return 3  # Advanced Stage

def HY43_classification(nhy):
    if nhy == 0:
        return 0  # Healthy
    elif nhy == 1:
        return 1  # Early Stage
    else:
        return 2  # Advanced Stage
    
def HY2_classification(nhy):
    if nhy == 0:
        return 0  # Healthy
    else:
        return 1  # Advanced Stage

def classification_mcid(x):
    if x <= -3:
        return 0  # Clinically meaningful improvement
    elif x >= 4:
        return 2   # Clinically meaningful worsening
    else:
        return 1   # Stable / no clinically meaningful change
    

# groupby para calcular estadísticas por paciente y visita

def compute_group_stats(df, group_level="PATNO", EVENT_ID:list=None):
    df = df.copy()
    df = df.loc[df["EVENT_ID"].isin(EVENT_ID),:]
    df.drop(columns=["EVENT_ID"], inplace=True)
    
    df_grouped = df.groupby(level=group_level)
    df_mean = df_grouped.mean().add_suffix("_mean")
    df_min  = df_grouped.min().add_suffix("_min")
    df_max  = df_grouped.max().add_suffix("_max")
    df_var  = df_grouped.var().add_suffix("_var")
    df_std  = df_grouped.std().add_suffix("_std")

    return pd.concat([df_mean, df_min, df_max, df_var, df_std], axis=1)
    



In [3]:
data['EVENT_ID'].unique()

array(['V06', 'V08', 'V10', 'V12'], dtype=object)

# V06 + Deltas

In [4]:
data_V10_delta = data.copy().set_index('PATNO')

V12_cols = ['ENRLLRRK2', 'ENRLGBA', 'COHORT_DEFINITION', 'SEX', 'RAWHITE', 'EDUCYRS', 'AGE_AT_VISIT', "NHY"]
updrs3_cols = [
    'NP3SPCH','NP3FACXP','NP3RIGN','NP3RIGRU','NP3RIGLU','NP3RIGRL','NP3RIGLL',
    'NP3FTAPR','NP3FTAPL','NP3HMOVR','NP3HMOVL','NP3PRSPR','NP3PRSPL','NP3TTAPR',
    'NP3TTAPL','NP3LGAGR','NP3LGAGL','NP3RISNG','NP3GAIT','NP3FRZGT','NP3PSTBL',
    'NP3POSTR','NP3BRADY','NP3PTRMR','NP3PTRML','NP3KTRMR','NP3KTRML','NP3RTARU',
    'NP3RTALU','NP3RTARL','NP3RTALL','NP3RTALJ','NP3RTCON'
]

data_last = data_V10_delta[V12_cols + ['EVENT_ID'] + updrs3_cols].copy()
data_last['UPDRS3_total'] = data_last[updrs3_cols].sum(axis=1)
data_last.drop(columns=updrs3_cols, inplace=True)

# Nueva visita objetivo: V12
data_V12 = data_last.loc[data_last["EVENT_ID"] == 'V12', :].copy()
# Última visita histórica: V10
data_V10 = data_last.loc[data_last["EVENT_ID"] == 'V10', :].copy()

data_V12 = data_V12.sort_index()
data_V10 = data_V10.sort_index()

data_V12['Delta_UPDRS3_V12_V10'] = data_V12['UPDRS3_total'] - data_V10['UPDRS3_total']

data_V12["STAGE_HY3"] = data_V12["NHY"].apply(HY3_classification)
data_V12["STAGE_HY2"] = data_V12["NHY"].apply(HY2_classification)
data_V12["STAGE_HY4"] = data_V12["NHY"].apply(HY4_classification)
data_V12["STAGE_HY43"] = data_V12["NHY"].apply(HY43_classification)
data_V12["STAGE_MCID"] = data_V12["Delta_UPDRS3_V12_V10"].apply(classification_mcid)

data_V12.drop(columns=["EVENT_ID", 'COHORT_DEFINITION', "NHY"], inplace=True)

data_V10_delta.drop(
    columns=['ENRLLRRK2', 'ENRLGBA', 'COHORT_DEFINITION', 'SEX', 'RAWHITE', 'EDUCYRS', 'AGE_AT_VISIT', "NHY"],
    inplace=True
)

# Generar los df para delta desplazados:
# antes: V04, V06, V08
# ahora: V06, V08, V10
V06_data = data_V10_delta.loc[data_V10_delta["EVENT_ID"] == 'V06', :].copy()
V08_data = data_V10_delta.loc[data_V10_delta["EVENT_ID"] == 'V08', :].copy()
V10_data = data_V10_delta.loc[data_V10_delta["EVENT_ID"] == 'V10', :].copy()

delta_V08_V06 = compute_delta(V06_data, V08_data, "V08_V06")
delta_V10_V08 = compute_delta(V08_data, V10_data, "V10_V08")
delta_V10_V06 = compute_delta(V06_data, V10_data, "V10_V06")

# Last visit hist: ahora V10
data_v10 = data_V10_delta.loc[data_V10_delta["EVENT_ID"] == 'V10', :].copy()
data_v10.columns = data_v10.columns + "_V10"
data_v10.drop(columns=["EVENT_ID_V10"], inplace=True)

# Unir los df de delta con el df de la última visita
data_V10_delta = data_V12.merge(data_v10, left_index=True, right_index=True, how='inner')
data_V10_delta = data_V10_delta.merge(delta_V08_V06, left_index=True, right_index=True, how='inner')
data_V10_delta = data_V10_delta.merge(delta_V10_V08, left_index=True, right_index=True, how='inner')
data_V10_delta = data_V10_delta.merge(delta_V10_V06, left_index=True, right_index=True, how='inner')

# Guardado
data_V10_delta.drop(
    columns=["STAGE_HY3", "STAGE_HY2", "STAGE_HY4", "STAGE_HY43", "STAGE_MCID", "Delta_UPDRS3_V12_V10", "UPDRS3_total"]
).to_csv(project_root / 'SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/X_V10+Deltas_TEST2.csv')

data_V10_delta["STAGE_HY3"].to_csv(project_root / 'SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/y_HY3_TEST2.csv')
data_V10_delta["STAGE_HY2"].to_csv(project_root / 'SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/y_HY2_TEST2.csv')
data_V10_delta["STAGE_HY4"].to_csv(project_root / 'SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/y_HY4_TEST2.csv')
data_V10_delta["STAGE_HY43"].to_csv(project_root / 'SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/y_HY43_TEST2.csv')
data_V10_delta["STAGE_MCID"].to_csv(project_root / 'SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/UPDRS3/FULL_FEATURES/y_MCID_TEST2.csv')
data_V10_delta['Delta_UPDRS3_V12_V10'].to_csv(project_root / 'SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/UPDRS3/FULL_FEATURES/y_delta_UPDRS3_V12_V10_TEST2.csv')
data_V10_delta['UPDRS3_total'].to_csv(project_root / 'SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/UPDRS3/FULL_FEATURES/y_UPDRS3_total_TEST2.csv')

data=pd.read_csv((project_root / 'SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/X_V10+Deltas_TEST2.csv'))
data

,PATNO,ENRLLRRK2,ENRLGBA,SEX,RAWHITE,EDUCYRS,AGE_AT_VISIT,MCAALTTM_V10,MCACUBE_V10,MCACLCKC_V10,...,NP3PTRMR_delta_V10_V06,NP3PTRML_delta_V10_V06,NP3KTRMR_delta_V10_V06,NP3KTRML_delta_V10_V06,NP3RTARU_delta_V10_V06,NP3RTALU_delta_V10_V06,NP3RTARL_delta_V10_V06,NP3RTALL_delta_V10_V06,NP3RTALJ_delta_V10_V06,NP3RTCON_delta_V10_V06
0,3051,0,0,1.0,1.0,18.0,76.7,1.0,1.0,0.0,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
1,3054,0,0,1.0,1.0,16.0,79.3,0.0,1.0,1.0,...,0.0,0.0,0.0,0.0,-2.0,0.0,0.0,1.0,0.0,0.0
2,3056,0,0,1.0,1.0,16.0,60.7,0.0,1.0,1.0,...,0.0,0.0,0.0,-1.0,1.0,1.0,1.0,0.0,0.0,1.0
3,3058,0,0,1.0,1.0,19.0,84.9,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,-2.0,0.0,0.0,0.0,0.0,-1.0
4,3060,0,0,1.0,1.0,16.0,80.1,1.0,1.0,1.0,...,-1.0,-1.0,0.0,0.0,-1.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
388,91097,0,0,1.0,0.0,16.0,72.3,1.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
389,100001,0,0,1.0,1.0,16.0,72.3,1.0,0.0,1.0,...,-1.0,0.0,0.0,-1.0,0.0,1.0,0.0,0.0,0.0,0.0
390,100006,0,0,0.0,1.0,15.0,60.7,1.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,-2.0,0.0,0.0,0.0
391,100007,0,0,1.0,1.0,19.0,72.3,1.0,1.0,1.0,...,1.0,1.0,0.0,0.0,-1.0,-1.0,0.0,0.0,0.0,-1.0


# V06 + Stats(v04+V04)

In [15]:
data_V10_stats = data.copy().set_index('PATNO')

V12_cols = ['ENRLLRRK2', 'ENRLGBA', 'SEX', 'RAWHITE', 'EDUCYRS', 'AGE_AT_VISIT']
data_V12 = data_V10_stats[V12_cols + ['EVENT_ID']].copy()
data_V12 = data_V12.loc[data_V12["EVENT_ID"] == 'V12', :].copy()  # New Visit
data_V12.drop(columns=["EVENT_ID"], inplace=True)

data_V10_stats.drop(
    columns=['ENRLLRRK2', 'ENRLGBA', 'COHORT_DEFINITION', 'SEX', 'RAWHITE', 'EDUCYRS', 'AGE_AT_VISIT', "NHY"],
    inplace=True
)

# Generar los df para estadísticas
# antes: entre V04 y V06
# ahora: entre V06 y V08
stats = compute_group_stats(data_V10_stats, group_level="PATNO", EVENT_ID=['V06', 'V08'])

# última visita histórica: antes V08, ahora V10
data_v10 = data_V10_stats.loc[data_V10_stats["EVENT_ID"] == 'V10', :].copy()
data_v10 = data_v10.add_suffix("_V10")
data_v10.drop(columns=["EVENT_ID_V10"], inplace=True)

data_V10_stats = data_V12.merge(data_v10, left_index=True, right_index=True, how='inner')
data_V10_stats = data_V10_stats.merge(stats, left_index=True, right_index=True, how='inner')

data_V10_stats.to_csv(
    project_root / 'SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/X_V10+STATS_TEST2.csv'
)

data_V10_stats.head()

,ENRLLRRK2,ENRLGBA,SEX,RAWHITE,EDUCYRS,AGE_AT_VISIT,MCAALTTM_V10,MCACUBE_V10,MCACLCKC_V10,MCACLCKN_V10,...,NP3PTRMR_std,NP3PTRML_std,NP3KTRMR_std,NP3KTRML_std,NP3RTARU_std,NP3RTALU_std,NP3RTARL_std,NP3RTALL_std,NP3RTALJ_std,NP3RTCON_std
PATNO,,,,,,,,,,,,,,,,,,,,,
3051,0,0,1.0,1.0,18.0,76.7,1.0,1.0,0.0,1.0,...,0.0,0.0,0.0,0.000000,0.707107,0.0,0.000000,0.000000,0.0,0.0
3054,0,0,1.0,1.0,16.0,79.3,0.0,1.0,1.0,1.0,...,0.0,0.0,0.0,0.000000,0.707107,0.0,0.000000,0.000000,0.0,0.0
3056,0,0,1.0,1.0,16.0,60.7,0.0,1.0,1.0,1.0,...,0.0,0.0,0.0,0.707107,0.000000,0.0,0.707107,0.707107,0.0,0.0
3058,0,0,1.0,1.0,19.0,84.9,0.0,0.0,1.0,1.0,...,0.0,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.0
3060,0,0,1.0,1.0,16.0,80.1,1.0,1.0,1.0,1.0,...,0.0,0.0,0.0,0.000000,0.707107,0.0,0.000000,0.000000,0.0,0.0


# Stats(BL+V04+V06)

In [16]:
data_stats = data.copy().set_index('PATNO')

V12_cols = ['ENRLLRRK2', 'ENRLGBA', 'SEX', 'RAWHITE', 'EDUCYRS', 'AGE_AT_VISIT']
data_V12 = data_stats[V12_cols + ['EVENT_ID']].copy()
data_V12 = data_V12.loc[data_V12["EVENT_ID"] == 'V12', :].copy()  # New Visit
data_V12.drop(columns=["EVENT_ID"], inplace=True)

data_stats.drop(
    columns=['ENRLLRRK2', 'ENRLGBA', 'COHORT_DEFINITION', 'SEX', 'RAWHITE', 'EDUCYRS', 'AGE_AT_VISIT', "NHY"],
    inplace=True
)

# Generar los df para estadísticas
# antes: V04, V06, V08
# ahora: V06, V08, V10
stats = compute_group_stats(data_stats, group_level="PATNO", EVENT_ID=['V06', 'V08', 'V10'])

data_stats = data_V12.merge(stats, left_index=True, right_index=True, how='inner')
data_stats.to_csv(project_root / 'SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/X_STATS_TEST2.csv')
data_stats.head()

,ENRLLRRK2,ENRLGBA,SEX,RAWHITE,EDUCYRS,AGE_AT_VISIT,MCAALTTM_mean,MCACUBE_mean,MCACLCKC_mean,MCACLCKN_mean,...,NP3PTRMR_std,NP3PTRML_std,NP3KTRMR_std,NP3KTRML_std,NP3RTARU_std,NP3RTALU_std,NP3RTARL_std,NP3RTALL_std,NP3RTALJ_std,NP3RTCON_std
PATNO,,,,,,,,,,,,,,,,,,,,,
3051,0,0,1.0,1.0,18.0,76.7,0.666667,1.000000,0.666667,1.0,...,0.57735,0.00000,0.0,0.00000,0.577350,0.00000,0.00000,0.00000,0.0,0.57735
3054,0,0,1.0,1.0,16.0,79.3,0.333333,0.666667,1.000000,1.0,...,0.00000,0.00000,0.0,0.00000,1.000000,0.00000,0.00000,0.57735,0.0,0.00000
3056,0,0,1.0,1.0,16.0,60.7,0.666667,1.000000,1.000000,1.0,...,0.00000,0.00000,0.0,0.57735,0.577350,0.57735,0.57735,0.57735,0.0,0.57735
3058,0,0,1.0,1.0,19.0,84.9,0.333333,0.666667,1.000000,1.0,...,0.00000,0.00000,0.0,0.00000,1.154701,0.00000,0.00000,0.00000,0.0,0.57735
3060,0,0,1.0,1.0,16.0,80.1,1.000000,1.000000,1.000000,1.0,...,0.57735,0.57735,0.0,0.00000,0.577350,0.00000,0.00000,0.00000,0.0,0.00000
